In [3]:
!pip install biopython pandas

In [4]:
from Bio import SeqIO

file_path = "../dataset/uniprot_sprot.fasta/uniprot_sprot.fasta"

records = list(SeqIO.parse(file_path, "fasta"))

print("Total sequences:", len(records))
print(records[0])

Total sequences: 574627
ID: sp|Q6GZX4|001R_FRG3G
Name: sp|Q6GZX4|001R_FRG3G
Description: sp|Q6GZX4|001R_FRG3G Putative transcription factor 001R OS=Frog virus 3 (isolate Goorha) OX=654924 GN=FV3-001R PE=4 SV=1
Number of features: 0
Seq('MAFSAEDVLKEYDRRRRMEALLLSLYYPNDRKLLDYKEWSPPRVQVECPKAPVE...TPL')


In [5]:
import pandas as pd

data = []

for record in records:
    sequence = str(record.seq)
    description = record.description.lower()
    
    if "enzyme" in description or "kinase" in description or "transferase" in description:
        label = "enzyme"
    elif "binding" in description:
        label = "binding"
    elif "transport" in description:
        label = "transport"
    else:
        label = "other"
    
    data.append([sequence, label])

df = pd.DataFrame(data, columns=["sequence", "label"])

df.head()

,sequence,label
0,MAFSAEDVLKEYDRRRRMEALLLSLYYPNDRKLLDYKEWSPPRVQV...,other
1,MSIIGATRLQNDKSDTYSAGPCYAGGCSAFTPRGTCGKDWDLGEQT...,other
2,MASNTVSAQGGSNRPVRDFSNIQDVAQFLLFDPIWNEQPGSIVPWK...,other
3,MYQAINPCPQSWYGSPQLEREIVCKMSGAPHYPNYYPVHPNALGGA...,other
4,MARPLLGKTSSVRRRLESLSACSIFFFLRKFCQKMASLVFLNSPVY...,other


In [6]:
df["label"].value_counts()

label
other        474907
enzyme        76678
binding       15112
transport      7930
Name: count, dtype: int64

In [7]:
df.to_csv("../dataset/protein_data.csv", index=False)

In [8]:
df = df[df["label"] != "other"]
df["label"].value_counts()

label
enzyme       76678
binding      15112
transport     7930
Name: count, dtype: int64

In [9]:
min_count = df["label"].value_counts().min()

df_balanced = df.groupby("label").sample(n=min_count, random_state=42)

df_balanced["label"].value_counts()

label
binding      7930
enzyme       7930
transport    7930
Name: count, dtype: int64

In [10]:
df_balanced.to_csv("../dataset/protein_balanced.csv", index=False)

In [11]:
import pandas as pd

df = pd.read_csv("../dataset/protein_balanced.csv")

df.head()

,sequence,label
0,MSDQFNSREARRKANSKSSPSPKKGKKRKKGGLFKKTLFTLLILFV...,binding
1,MSEPLLHLTGISRSFTAGDREFLALKHIDLSIQAGEMVAITGASGS...,binding
2,MKVNPNNIELIISAVKEEQYPETELSEVALSGRSNVGKSTFINSMI...,binding
3,MARYDLVDRLNTTFRQMEQELAAFAAHLEQHKLLVARVFSLPEVKK...,binding
4,MFWGIEISKVPVKFTPAFDLHITTACLSAVAKDTGRNVLQVKYDGK...,binding


In [12]:
def kmer_sequence(seq, k=3):
    return " ".join([seq[i:i+k] for i in range(len(seq)-k+1)])

df["kmer"] = df["sequence"].apply(lambda x: kmer_sequence(x, k=3))

df.head()

,sequence,label,kmer
0,MSDQFNSREARRKANSKSSPSPKKGKKRKKGGLFKKTLFTLLILFV...,binding,MSD SDQ DQF QFN FNS NSR SRE REA EAR ARR RRK RK...
1,MSEPLLHLTGISRSFTAGDREFLALKHIDLSIQAGEMVAITGASGS...,binding,MSE SEP EPL PLL LLH LHL HLT LTG TGI GIS ISR SR...
2,MKVNPNNIELIISAVKEEQYPETELSEVALSGRSNVGKSTFINSMI...,binding,MKV KVN VNP NPN PNN NNI NIE IEL ELI LII IIS IS...
3,MARYDLVDRLNTTFRQMEQELAAFAAHLEQHKLLVARVFSLPEVKK...,binding,MAR ARY RYD YDL DLV LVD VDR DRL RLN LNT NTT TT...
4,MFWGIEISKVPVKFTPAFDLHITTACLSAVAKDTGRNVLQVKYDGK...,binding,MFW FWG WGI GIE IEI EIS ISK SKV KVP VPV PVK VK...


In [13]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()

X = vectorizer.fit_transform(df["kmer"])
y = df["label"]

In [14]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [15]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

     binding       0.86      0.91      0.88      1563
      enzyme       0.88      0.86      0.87      1573
   transport       0.93      0.91      0.92      1622

    accuracy                           0.89      4758
   macro avg       0.89      0.89      0.89      4758
weighted avg       0.89      0.89      0.89      4758



In [16]:
print(X.shape)

(23790, 8189)


In [ ]:
print(y.shape)

# Model Improvements

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(df["kmer"])

In [19]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)

In [20]:
from sklearn.linear_model import LogisticRegression

model_lr = LogisticRegression(max_iter=1000)
model_lr.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [21]:
from sklearn.metrics import classification_report

y_pred = model_lr.predict(X_test)
print("TF-IDF + Logistic Regression")
print(classification_report(y_test, y_pred))

TF-IDF + Logistic Regression
              precision    recall  f1-score   support

     binding       0.89      0.90      0.89      1563
      enzyme       0.87      0.89      0.88      1573
   transport       0.94      0.91      0.93      1622

    accuracy                           0.90      4758
   macro avg       0.90      0.90      0.90      4758
weighted avg       0.90      0.90      0.90      4758



In [22]:
from sklearn.ensemble import RandomForestClassifier

model_rf = RandomForestClassifier(n_estimators=100)
model_rf.fit(X_train, y_train)

y_pred_rf = model_rf.predict(X_test)

print("Random Forest")
print(classification_report(y_test, y_pred_rf))

Random Forest
              precision    recall  f1-score   support

     binding       0.94      0.84      0.89      1563
      enzyme       0.81      0.90      0.86      1573
   transport       0.91      0.91      0.91      1622

    accuracy                           0.88      4758
   macro avg       0.89      0.88      0.88      4758
weighted avg       0.89      0.88      0.88      4758



In [23]:
from sklearn.svm import LinearSVC

model_svm = LinearSVC()
model_svm.fit(X_train, y_train)

y_pred_svm = model_svm.predict(X_test)

print("SVM")
print(classification_report(y_test, y_pred_svm))

SVM
              precision    recall  f1-score   support

     binding       0.89      0.92      0.90      1563
      enzyme       0.90      0.88      0.89      1573
   transport       0.94      0.93      0.94      1622

    accuracy                           0.91      4758
   macro avg       0.91      0.91      0.91      4758
weighted avg       0.91      0.91      0.91      4758



In [24]:
import joblib

joblib.dump(model_svm, "../models/best_model.pkl")
joblib.dump(tfidf, "../models/vectorizer.pkl")

['../models/vectorizer.pkl']

In [25]:
def predict_protein(sequence):
    # convert to k-mer
    kmer_seq = " ".join([sequence[i:i+3] for i in range(len(sequence)-2)])
    
    # transform using saved vectorizer
    X_input = tfidf.transform([kmer_seq])
    
    # predict
    prediction = model_svm.predict(X_input)[0]
    
    return prediction

In [31]:
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report

base_svm = LinearSVC()

model_svm_prob = CalibratedClassifierCV(base_svm, cv=3)

model_svm_prob.fit(X_train, y_train)

y_pred = model_svm_prob.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

     binding       0.90      0.92      0.91      1563
      enzyme       0.90      0.89      0.89      1573
   transport       0.94      0.93      0.94      1622

    accuracy                           0.91      4758
   macro avg       0.91      0.91      0.91      4758
weighted avg       0.91      0.91      0.91      4758



In [32]:
def predict_protein_with_confidence(sequence):
    kmer_seq = " ".join([sequence[i:i+3] for i in range(len(sequence)-2)])
    
    X_input = tfidf.transform([kmer_seq])
    
    prediction = model_svm_prob.predict(X_input)[0]
    
    probabilities = model_svm_prob.predict_proba(X_input)[0]
    confidence = max(probabilities) * 100
    
    return prediction, round(confidence, 2)

In [33]:
test_seq = df["sequence"].iloc[0]

prediction, confidence = predict_protein_with_confidence(test_seq)

print("Prediction:", prediction)
print("Confidence:", confidence, "%")

Prediction: binding
Confidence: 91.72 %


In [34]:
import joblib

joblib.dump(model_svm_prob, "../models/best_model_with_confidence.pkl")
joblib.dump(tfidf, "../models/vectorizer.pkl")

['../models/vectorizer.pkl']

In [46]:
new_seq = "KELD"

prediction, confidence = predict_protein_with_confidence(new_seq)

print("Prediction:", prediction)
print("Confidence:", confidence, "%")

Prediction: binding
Confidence: 86.07 %


In [48]:
new_seq = "MGLSDGEWQLVLNVWGKVEADIPGHGQEVLIRLFKSHPETLEKFDRFKHLKTEAEMKASEDLKKHGATVLTALGGILKKKGHHEAEIKPLAQSHATKHKIPVKYLEFISEAIIHVLHSRHPGDFGADAQGAMNKALELFRKDIAAKYKELGFQG"

prediction, confidence = predict_protein_with_confidence(new_seq)

print(prediction, confidence)

binding 70.66


In [49]:
from sklearn.metrics import accuracy_score, f1_score

results = []

models = {
    "TF-IDF + Logistic Regression": model_lr,
    "TF-IDF + Random Forest": model_rf,
    "TF-IDF + SVM": model_svm,
    "TF-IDF + SVM with Confidence": model_svm_prob
}

for name, m in models.items():
    pred = m.predict(X_test)
    acc = accuracy_score(y_test, pred)
    f1 = f1_score(y_test, pred, average="weighted")
    
    results.append([name, acc, f1])

results_df = pd.DataFrame(results, columns=["Model", "Accuracy", "Weighted F1"])
results_df

,Model,Accuracy,Weighted F1
0,TF-IDF + Logistic Regression,0.901009,0.901180
1,TF-IDF + Random Forest,0.883144,0.883699
2,TF-IDF + SVM,0.911517,0.911535
3,TF-IDF + SVM with Confidence,0.911728,0.911783


In [50]:
import joblib

joblib.dump(model_svm_prob, "../models/final_protein_model.pkl")
joblib.dump(tfidf, "../models/final_vectorizer.pkl")

['../models/final_vectorizer.pkl']

In [51]:
def predict_protein_function(sequence):
    sequence = sequence.upper().replace(" ", "").replace("\n", "")
    
    kmer_seq = " ".join([sequence[i:i+3] for i in range(len(sequence)-2)])
    X_input = tfidf.transform([kmer_seq])
    
    prediction = model_svm_prob.predict(X_input)[0]
    probabilities = model_svm_prob.predict_proba(X_input)[0]
    confidence = max(probabilities) * 100
    
    return prediction, round(confidence, 2)

sample_seq = df["sequence"].sample(1).values[0]

pred, conf = predict_protein_function(sample_seq)

print("Prediction:", pred)
print("Confidence:", conf, "%")

Prediction: enzyme
Confidence: 92.09 %
